# ELI Index Runner — E3SM Hindcast (ne30pg2 / MPAS-Ocean)

This notebook computes the **Equatorial Longitude Index (ELI)** — the
area-weighted centroid longitude of warm SST (SST > tropical-mean SST) in
the equatorial Pacific — for every E3SM S2D hindcast case and ensemble member,
then saves the results as lead-indexed NetCDF files.

**Pipeline:** MPAS-Ocean `timeSeriesStatsMonthly` → ELI monthly time series → NetCDF

**Output layout:** `OUTDIR/E3SMLE{mm}_ELI_N{nens}_M{nlead}.nc`  
Dimensions: `(Y=init_year, L=lead_month, M=member)`

Run this notebook first to generate the ELI index files used by
`8_refactor_eli_index.ipynb` for skill analysis and plotting.

In [1]:
import os
import sys
# Resolve native-library data from the interpreter running this kernel.
_env_prefix = sys.prefix
_proj_path = os.path.join(_env_prefix, "share", "proj")
if os.path.isfile(os.path.join(_proj_path, "proj.db")):
    os.environ["CONDA_PREFIX"] = _env_prefix
    os.environ["PROJ_LIB"]    = _proj_path
    os.environ["PROJ_DATA"]   = _proj_path

import glob
import warnings
from pathlib import Path

import numpy as np
import xarray as xr

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

from esp_lab import data_access_e3sm as data_access

print(f"Python      : {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set)')}")

Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


Python      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/homes/z/zhan391/.conda/envs/e3sm_analysis


Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


## Configuration — fill in ALL fields before running

Fields left as `None` will cause the **Validate** cell to raise an error.

In [2]:
# ------------------------------------------------------------------ #
#  PATHS  (all required)
# ------------------------------------------------------------------ #

# Root simulation directory containing case subdirectories.
SIM_DIR    = Path("/global/cfs/cdirs/e3smdata/simulations/S2S2D")

# Where to write processed ELI index files.
OUTDIR     = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE")

# MPAS-Ocean mesh file (for cell lat/lon/area).
MESH_FILE  = Path(
    "/global/cfs/cdirs/e3sm/inputdata/ocn/mpas-o/IcoswISC30E3r5/"
    "mpaso.IcoswISC30E3r5.rstFromG-chrysalis.20231121.nc"
)

# ------------------------------------------------------------------ #
#  CASES — built from prefix + init tags
# ------------------------------------------------------------------ #

CASE_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL"

START_YEAR  = 1980
END_YEAR    = 2018
EXCL_YEAR   = None        # set to an int to exclude one year, or None

INIT_MONTHS = [5, 11]     # May and November starts

lead_years = [y for y in np.arange(START_YEAR, END_YEAR + 1) if y != EXCL_YEAR]

CASES = [
    f"{CASE_PREFIX}_{year}{init_month:02d}0100"
    for init_month in INIT_MONTHS
    for year in lead_years
]

print(f"Total cases : {len(CASES)}")
print("First few   :", CASES[:3])
print("Last few    :", CASES[-3:])

# ------------------------------------------------------------------ #
#  ENSEMBLE
# ------------------------------------------------------------------ #

CASE_NENS  = 10
MEMBERS    = [f"EN{i:02d}" for i in range(CASE_NENS)]
NENS       = None    # e.g. 3 for a quick test; None = all

# ------------------------------------------------------------------ #
#  ELI PARAMETERS
# ------------------------------------------------------------------ #

# Equatorial Pacific region for centroid calculation.
ELI_LAT_MIN  = -5.0
ELI_LAT_MAX  =  5.0
ELI_LON_MIN  = 120.0   # degrees East
ELI_LON_MAX  = 290.0   # degrees East (= 70°W)

# Tropical-mean latitude band for reference temperature Tc.
TC_LAT_HALF  = 5.0     # ±5° around equator

# MPAS-Ocean history stream filename pattern.
OCN_HIST_PATTERN = "*mpaso.hist.am.timeSeriesStatsMonthly.*.nc"

# SST variable in MPAS-Ocean monthly files (°C in MPAS).
OCN_SST_VAR  = "timeMonthly_avg_activeTracers_temperature"

NLEAD        = 24       # monthly lead times to save

# ------------------------------------------------------------------ #
#  RUN CONTROL
# ------------------------------------------------------------------ #

FORCE_REWRITE = True    # overwrite existing output files
NWORKERS      = 12       # member-level I/O workers; use 1 to disable parallel reads

print(f"\nNLEAD       : {NLEAD}")
print(f"CASE_NENS   : {CASE_NENS}")
print(f"FORCE_REWRITE: {FORCE_REWRITE}")
print(f"NWORKERS    : {NWORKERS}")

Total cases : 78
First few   : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100']
Last few    : ['WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2016110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2017110100', 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100']

NLEAD       : 24
CASE_NENS   : 10
FORCE_REWRITE: True
NWORKERS    : 12


## Validate — run this before anything else

In [3]:
# ---- enforce all required fields are set ----
_required = {
    "SIM_DIR":    SIM_DIR,
    "OUTDIR":     OUTDIR,
    "MESH_FILE":  MESH_FILE,
    "CASES":      CASES,
    "MEMBERS":    MEMBERS,
}
_missing = [k for k, v in _required.items() if v is None]
if _missing:
    raise ValueError(
        "The following required fields are still None — fill them in the "
        "Configuration cell:\n" + "\n".join(f"  {k}" for k in _missing)
    )

# ---- basic configuration checks ----
_errors = []
if not CASES:
    _errors.append("CASES is empty; check START_YEAR, END_YEAR, EXCL_YEAR, and INIT_MONTHS.")
if not lead_years:
    _errors.append("lead_years is empty; check START_YEAR, END_YEAR, and EXCL_YEAR.")
if NLEAD <= 0:
    _errors.append(f"NLEAD must be positive, got {NLEAD}.")
if CASE_NENS <= 0:
    _errors.append(f"CASE_NENS must be positive, got {CASE_NENS}.")
if NENS is not None and NENS <= 0:
    _errors.append(f"NENS must be positive or None, got {NENS}.")
if NWORKERS <= 0:
    _errors.append(f"NWORKERS must be positive, got {NWORKERS}.")

# ---- check paths ----
if not SIM_DIR.is_dir():
    _errors.append(f"SIM_DIR does not exist: {SIM_DIR}")
if not MESH_FILE.is_file():
    _errors.append(f"MESH_FILE does not exist: {MESH_FILE}")
if _errors:
    raise ValueError("Configuration validation failed:\n" + "\n".join(_errors))

OUTDIR.mkdir(parents=True, exist_ok=True)
if not os.access(OUTDIR, os.W_OK):
    raise PermissionError(f"OUTDIR is not writable: {OUTDIR}")

# ---- resolve member list ----
_ref_case_dir = SIM_DIR / CASES[0]
if not _ref_case_dir.is_dir():
    raise FileNotFoundError(f"First case directory does not exist: {_ref_case_dir}")

_all_members = sorted(p.name for p in _ref_case_dir.iterdir()
                      if p.is_dir() and p.name.startswith("EN"))
if not _all_members:
    raise ValueError(f"No EN* member directories found in {_ref_case_dir}")

if MEMBERS:
    missing_members = sorted(set(MEMBERS) - set(_all_members))
    if missing_members:
        raise ValueError(
            "Requested member directories are missing in the reference case "
            f"{_ref_case_dir}: {missing_members}"
        )
    members_avail = list(MEMBERS)
else:
    members_avail = _all_members

if NENS is not None:
    if NENS > len(members_avail):
        raise ValueError(f"NENS={NENS} exceeds available members ({len(members_avail)}).")
    members_avail = members_avail[:NENS]

RUN_NENS = len(members_avail)
if RUN_NENS == 0:
    raise ValueError("No members selected for processing.")

# Keep output filenames explicit about the ensemble size actually processed.
OUT_NENS = RUN_NENS

print("Configuration valid ✓")
print(f"  SIM_DIR   : {SIM_DIR}")
print(f"  OUTDIR    : {OUTDIR}")
print(f"  MESH_FILE : {MESH_FILE}")
print(f"  Cases     : {len(CASES)} total  ({CASES[0]}  ...  {CASES[-1]})")
print(f"  Members   : {members_avail}  (RUN_NENS={RUN_NENS}, NENS={NENS})")


Configuration valid ✓
  SIM_DIR   : /global/cfs/cdirs/e3smdata/simulations/S2S2D
  OUTDIR    : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE
  MESH_FILE : /global/cfs/cdirs/e3sm/inputdata/ocn/mpas-o/IcoswISC30E3r5/mpaso.IcoswISC30E3r5.rstFromG-chrysalis.20231121.nc
  Cases     : 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100)
  Members   : ['EN00', 'EN01', 'EN02', 'EN03', 'EN04', 'EN05', 'EN06', 'EN07', 'EN08', 'EN09']  (RUN_NENS=10, NENS=None)


## 1.  Survey available ocean files

In [4]:
# Survey MPAS-Ocean monthly file counts for the first two cases.
for case in CASES[:2]:
    case_dir = SIM_DIR / case
    print(f"\nCase : {case}")
    for member in members_avail:
        hist_dir = case_dir / member / "archive" / "ocn" / "hist"
        ocn_files = sorted(hist_dir.glob(OCN_HIST_PATTERN))
        print(f"  {member}: {len(ocn_files)} files", end="")
        if ocn_files:
            print(f"  [{ocn_files[0].name}  ...  {ocn_files[-1].name}]")
        else:
            print("  <none found>")


Case : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
  EN00: 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.mpaso.hist.am.timeSeriesStatsMonthly.1980-05-01.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.mpaso.hist.am.timeSeriesStatsMonthly.1982-04-01.nc]
  EN01: 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.mpaso.hist.am.timeSeriesStatsMonthly.1980-05-01.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN01.mpaso.hist.am.timeSeriesStatsMonthly.1982-04-01.nc]
  EN02: 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN02.mpaso.hist.am.timeSeriesStatsMonthly.1980-05-01.nc  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN02.mpaso.hist.am.timeSeriesStatsMonthly.1982-04-01.nc]
  EN03: 24 files  [WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN03.mpaso.hist.am.timeSeriesStatsMonthly.1980-05-01.nc  ...  WCYCL20TR

## 2.  Verify ocean variables in MPAS-Ocean files

The ELI computation requires `timeMonthly_avg_activeTracers_temperature` at
`nVertLevels=0` (surface layer).  This cell opens one file and checks for the
variable.

In [5]:
# Verify SST variable in the first available case/member.
_case = CASES[0]
_member = members_avail[0]
_hist_dir = SIM_DIR / _case / _member / "archive" / "ocn" / "hist"
_ocn_files = sorted(_hist_dir.glob(OCN_HIST_PATTERN))

assert _ocn_files, f"No ocean files found in {_hist_dir}"

_ds = xr.open_dataset(_ocn_files[0], decode_times=False)
_found    = OCN_SST_VAR in _ds.data_vars
_has_vert = "nVertLevels" in _ds[OCN_SST_VAR].dims if _found else False
_ds.close()

print(f"Case   : {_case}")
print(f"Member : {_member}")
print(f"File   : {_ocn_files[0].name}")
print(f"  {OCN_SST_VAR}: {'present ✓' if _found else 'MISSING <-- problem'}")
if _found:
    print(f"  nVertLevels dim: {'present ✓' if _has_vert else 'absent (scalar surface?)'} ")

if not _found:
    raise KeyError(f"{OCN_SST_VAR!r} not found — check OCN_SST_VAR in the Config cell.")

Case   : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100
Member : EN00
File   : WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100.EN00.mpaso.hist.am.timeSeriesStatsMonthly.1980-05-01.nc
  timeMonthly_avg_activeTracers_temperature: present ✓
  nVertLevels dim: present ✓ 


## 3.  Load mesh geometry

Extract cell-centre latitudes, longitudes, and areas from the MPAS-Ocean mesh.
Masks for the equatorial Pacific and broad tropical band are precomputed here
and reused for every case/member.

In [6]:
with xr.open_dataset(MESH_FILE) as mesh:
    for mesh_var in ["latCell", "lonCell", "areaCell"]:
        if mesh_var not in mesh:
            raise KeyError(f"Required mesh variable {mesh_var!r} is missing from {MESH_FILE}")

    lat  = mesh["latCell"].values  * 180.0 / np.pi   # degrees North
    lon  = mesh["lonCell"].values  * 180.0 / np.pi   # degrees East  [0, 360]
    area = mesh["areaCell"].values                    # m²

lon = np.mod(lon, 360.0)

if not (np.isfinite(lat).all() and np.isfinite(lon).all()):
    raise ValueError("Mesh latitude/longitude contains non-finite values.")
if not (np.isfinite(area).all() and np.all(area > 0.0)):
    raise ValueError("Mesh cell areas must be finite and strictly positive.")

# Equatorial Pacific region for ELI centroid.
region_eq = (
    (lat >= ELI_LAT_MIN) & (lat <= ELI_LAT_MAX) &
    (lon >= ELI_LON_MIN) & (lon <= ELI_LON_MAX)
)

# Broad tropical band for reference temperature Tc.
region_tropics = (lat >= -TC_LAT_HALF) & (lat <= TC_LAT_HALF)

if not np.any(region_eq):
    raise ValueError("Equatorial Pacific mask selected zero mesh cells; check ELI bounds.")
if not np.any(region_tropics):
    raise ValueError("Tropical reference mask selected zero mesh cells; check TC_LAT_HALF.")

# Integer indices are faster and cleaner for repeated file reads.
idx_union = np.flatnonzero(region_eq | region_tropics)
eq_on_union = region_eq[idx_union]
tropics_on_union = region_tropics[idx_union]

lon_eq  = lon[idx_union][eq_on_union].astype(np.float32)
area_eq = area[idx_union][eq_on_union].astype(np.float64)
area_tr = area[idx_union][tropics_on_union].astype(np.float64)

print(f"Mesh cells total    : {len(lat):,}")
print(f"Equatorial Pacific  : {region_eq.sum():,} cells  "
      f"(lat [{ELI_LAT_MIN}, {ELI_LAT_MAX}]°, lon [{ELI_LON_MIN}, {ELI_LON_MAX}]°)")
print(f"Tropical band       : {region_tropics.sum():,} cells  (lat ±{TC_LAT_HALF}°)")

Mesh cells total    : 465,044
Equatorial Pacific  : 24,009 cells  (lat [-5.0, 5.0]°, lon [120.0, 290.0]°)
Tropical band       : 42,941 cells  (lat ±5.0°)


## 4.  Compute ELI for all cases

For each init-month × init-year × member:
1. Load MPAS-Ocean monthly SST.
2. Compute tropical-mean SST $T_c$ (area-weighted over ±5°).
3. Identify warm-pool cells: SST > $T_c$ in the equatorial Pacific.
4. ELI = area-weighted centroid longitude of warm-pool cells.

Results are saved as `E3SMLE{mm:02d}_ELI_N{nens:02d}_M{nlead:02d}.nc`
with dimensions `(Y=init_year, L=lead_month, M=member)`.

In [ ]:
def _weighted_mean_skipna_1d(values: np.ndarray, weights: np.ndarray) -> float:
    """Area-weighted mean for one time slice, ignoring non-finite values."""
    finite = np.isfinite(values)
    if not np.any(finite):
        return np.nan
    denom = np.sum(weights[finite])
    if denom <= 0.0:
        return np.nan
    return float(np.sum(values[finite] * weights[finite]) / denom)


def _read_surface_sst_on_cells(path: Path, idx_union: np.ndarray) -> np.ndarray:
    """Read one monthly MPAS-Ocean file on the needed cells only."""
    with xr.open_dataset(path, decode_times=False) as ds:
        if OCN_SST_VAR not in ds:
            raise KeyError(f"{OCN_SST_VAR!r} not found in {path}")

        sst_da = ds[OCN_SST_VAR]
        if "nVertLevels" in sst_da.dims:
            sst_da = sst_da.isel(nVertLevels=0)
        if "Time" in sst_da.dims:
            sst_da = sst_da.isel(Time=0)
        if "nCells" not in sst_da.dims:
            raise ValueError(f"{OCN_SST_VAR!r} must have nCells dimension; found {sst_da.dims}")

        sst = sst_da.isel(nCells=idx_union).astype("float32").load().values

    if sst.ndim != 1:
        raise ValueError(f"Expected 1-D SST after slicing {path}, got shape {sst.shape}")
    return sst


def compute_eli_member(
    hist_dir: Path,
    pattern: str,
    idx_union: np.ndarray,
    eq_on_union: np.ndarray,
    tropics_on_union: np.ndarray,
    lon_eq: np.ndarray,
    area_eq: np.ndarray,
    area_tr: np.ndarray,
    nlead: int,
) -> np.ndarray:
    """
    Compute monthly ELI for one ensemble member.

    This reads files one-by-one instead of using ``open_mfdataset``. For this
    workflow that is usually faster because each member has a small fixed number
    of monthly files and only one variable/cell subset is needed.
    """
    files = sorted(hist_dir.glob(pattern))
    if not files:
        return np.full(nlead, np.nan, dtype=np.float32)
    if len(files) < nlead:
        print(f"  [WARN] {hist_dir}: only {len(files)} monthly files found; expected {nlead}.")

    out = np.full(nlead, np.nan, dtype=np.float32)
    for lead_idx, path in enumerate(files[:nlead]):
        sst = _read_surface_sst_on_cells(path, idx_union)
        sst_eq = sst[eq_on_union]
        sst_tr = sst[tropics_on_union]

        Tc = _weighted_mean_skipna_1d(sst_tr, area_tr)
        if not np.isfinite(Tc):
            continue

        warm = np.isfinite(sst_eq) & (sst_eq > Tc)
        den = np.sum(area_eq[warm])
        if den > 0.0:
            out[lead_idx] = np.sum(area_eq[warm] * lon_eq[warm]) / den

    return out


def _compute_case_member(args):
    yi, mi, case, member, hist_dir = args
    try:
        values = compute_eli_member(
            hist_dir, OCN_HIST_PATTERN,
            idx_union, eq_on_union, tropics_on_union,
            lon_eq, area_eq, area_tr,
            NLEAD,
        )
        return yi, mi, case, member, values, None
    except Exception as exc:
        return yi, mi, case, member, None, exc


for init_month in INIT_MONTHS:
    outfile = OUTDIR / f"E3SMLE{init_month:02d}_ELI_N{OUT_NENS:02d}_M{NLEAD:02d}.nc"

    if outfile.exists() and not FORCE_REWRITE:
        print(f"Skipping init month {init_month:02d} — output exists: {outfile}")
        continue

    print(f"\n=== Init month {init_month:02d} ===")
    t0 = time.time()

    eli_all = np.full(
        (len(lead_years), NLEAD, RUN_NENS),
        np.nan, dtype=np.float32,
    )

    cases_for_month = [
        f"{CASE_PREFIX}_{year}{init_month:02d}0100" for year in lead_years
    ]

    workers = min(int(NWORKERS), RUN_NENS)

    for yi, (year, case) in enumerate(zip(lead_years, cases_for_month)):
        case_dir = SIM_DIR / case
        if not case_dir.is_dir():
            print(f"  [WARN] Case directory not found: {case}")
            continue

        tasks = [
            (yi, mi, case, member, case_dir / member / "archive" / "ocn" / "hist")
            for mi, member in enumerate(members_avail)
        ]

        if workers == 1:
            results_iter = map(_compute_case_member, tasks)
        else:
            pool = ThreadPoolExecutor(max_workers=workers)
            futures = [pool.submit(_compute_case_member, task) for task in tasks]
            results_iter = (future.result() for future in as_completed(futures))

        try:
            for _, mi, case_name, member, values, exc in results_iter:
                if exc is not None:
                    print(f"  [ERROR] {case_name}/{member}: {exc}")
                    continue
                eli_all[yi, :, mi] = values
        finally:
            if workers != 1:
                pool.shutdown(wait=True)

        n_ok = int(np.sum(np.isfinite(eli_all[yi])))
        print(f"  [{yi + 1:3d}/{len(lead_years)}] {case}  — {n_ok}/{NLEAD * RUN_NENS} valid values")

    # ---- build output dataset ----
    ds_out = xr.Dataset(
        {
            "eli": xr.DataArray(
                eli_all,
                dims=("Y", "L", "M"),
                coords={
                    "Y": np.array(lead_years, dtype=np.int32),
                    "L": np.arange(1, NLEAD + 1, dtype=np.int32),
                    "M": np.arange(RUN_NENS, dtype=np.int32),
                },
                attrs={
                    "long_name": "Equatorial Longitude Index",
                    "units": "degrees_east",
                    "description": (
                        "Area-weighted centroid longitude of warm SST cells "
                        "(SST > tropical-mean SST) in the equatorial Pacific "
                        f"(lat {ELI_LAT_MIN}–{ELI_LAT_MAX}°, "
                        f"lon {ELI_LON_MIN}–{ELI_LON_MAX}°).  "
                        f"Tc reference band: ±{TC_LAT_HALF}°."
                    ),
                },
            )
        }
    )

    ds_out["Y"].attrs = {"long_name": "initialization year", "units": "year"}
    ds_out["L"].attrs = {"long_name": "forecast lead month", "units": "months"}
    ds_out["M"].attrs = {"long_name": "ensemble member index"}

    ds_out["member_id"] = xr.DataArray(
        np.array(members_avail, dtype="U5"),
        dims="M",
        attrs={"long_name": "ensemble member label"},
    )

    ds_out.attrs = {
        "case_prefix":  CASE_PREFIX,
        "init_month":   int(init_month),
        "start_year":   int(START_YEAR),
        "end_year":     int(END_YEAR),
        "case_nens":    int(CASE_NENS),
        "run_nens":     int(RUN_NENS),
        "nworkers":     int(workers),
        "eli_lat_min":  ELI_LAT_MIN,
        "eli_lat_max":  ELI_LAT_MAX,
        "eli_lon_min":  ELI_LON_MIN,
        "eli_lon_max":  ELI_LON_MAX,
        "tc_lat_half":  TC_LAT_HALF,
    }

    # Atomic write
    tmp_file = outfile.with_suffix(".tmp.nc")
    try:
        ds_out.to_netcdf(
            tmp_file,
            encoding={"eli": {"zlib": True, "complevel": 1, "dtype": "float32"}},
        )
        os.replace(tmp_file, outfile)
    finally:
        if tmp_file.exists():
            tmp_file.unlink()

    elapsed = time.time() - t0
    n_nan = int(np.isnan(eli_all).sum())
    n_tot = eli_all.size
    print(f"  Saved → {outfile}  ({elapsed:.0f}s, {n_nan}/{n_tot} NaN values)")



=== Init month 05 ===
  [  1/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  — 240/240 valid values


## 5.  Verify saved output

In [ ]:
import matplotlib.pyplot as plt

for init_month in INIT_MONTHS:
    outfile = OUTDIR / f"E3SMLE{init_month:02d}_ELI_N{OUT_NENS:02d}_M{NLEAD:02d}.nc"
    if not outfile.exists():
        print(f"[MISSING] {outfile}")
        continue

    with xr.open_dataset(outfile) as ds:
        eli = ds["eli"]
        n_valid = int(np.isfinite(eli).sum())
        n_total = int(eli.size)
        print(f"\nInit month {init_month:02d}  →  {outfile.name}")
        print(f"  dims   : {dict(eli.sizes)}")
        if n_valid:
            print(f"  min    : {float(eli.min(skipna=True)):.2f}°E")
            print(f"  max    : {float(eli.max(skipna=True)):.2f}°E")
            print(f"  mean   : {float(eli.mean(skipna=True)):.2f}°E")
        else:
            print("  min/max/mean: unavailable (all values are NaN)")
        print(f"  valid  : {n_valid}/{n_total}")

        # Quick ensemble-mean time series for the first init year
        eli_yr0 = eli.isel(Y=0).values  # (L, M)
        fig, ax = plt.subplots(figsize=(9, 3))
        ax.plot(np.arange(1, NLEAD + 1), eli_yr0, color="gray", linewidth=0.7, alpha=0.6)
        ax.plot(np.arange(1, NLEAD + 1), np.nanmean(eli_yr0, axis=-1),
                color="k", linewidth=2, label="Ens mean")
        ax.set_xlabel("Lead month")
        ax.set_ylabel("ELI (°E)")
        ax.set_title(
            f"ELI — init month {init_month:02d},  year {int(ds['Y'].isel(Y=0))}"
        )
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()
